# 第59章 联合分布图（jointplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 16 / 20 步：探索变量关系与趋势**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 回归图（regplot / lmplot）  →  **本章任务：** 联合分布图（jointplot）  →  **下一步：** 成对关系图（pairplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

只看单个变量就像只读剧本的一段独白，往往会掩盖两个变量之间的真实关系。


## 本章目标

学完本章，你将能够：

- **理解**：理解「联合分布图（jointplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「联合分布图（jointplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「联合分布图（jointplot）」并读出其中的结论。


## 适用场景
**背景引入**：只看单个变量就像只读剧本的一段独白，往往会掩盖两个变量之间的真实关系。联合分布图（jointplot）把一对数值变量同时画在中心主图和周围的边缘小图上，让你一眼看出它们是一起升高还是各走各路、谁更分散谁更集中，还能顺带发现离群点。做数据分析时，凡是怀疑两个数字指标互相影响——比如访问量是否带动了销售额、投入是否真的换来转化——都可以先用它探探路。

深入检查一对数值变量的联合关系和单变量分布。


## 数据结构

两列连续数值，可增加hue分类。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 kind="scatter" 改为 kind="kde" 或 kind="reg"，对比不同中心图类型的信息展示
2. 修改 height 参数从 6 改为 8，观察整体图形尺寸对可读性的影响
3. 调整 ratio 参数（如设为 3），说明主图与边缘图比例对布局的影响


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `sns.jointplot()`、`grid.set_axis_labels()`、`grid.fig.suptitle()`、`plt.show()` | 深入检查一对数值变量的联合关系和单变量分布。 | 只看中心趋势忽略边缘偏态 |
| 进阶变体 | `sns.jointplot()`、`grid.set_axis_labels()`、`grid.fig.suptitle()`、`plt.show()` | 在基础图表上增加分组、注释、布局或交互 | 大样本散点过度重叠 |
| 关键参数 | `kind` | scatter/hex/kde/reg/hist | 只看中心趋势忽略边缘偏态 |
| 关键参数 | `marginal_kws` | 边缘设置 | 大样本散点过度重叠 |
| 关键参数 | `height` | 尺寸 | 不适合一次比较很多变量 |
| 关键参数 | `ratio` | 主图比例 | 只看中心趋势忽略边缘偏态 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-59 -->
### 数学推导｜联合分布与协方差矩阵

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜中心化每个变量。** $x_i^c=x_i-\bar x$、$y_i^c=y_i-\bar y$。

**第 2 步｜观察离差乘积。** 同向偏离产生正乘积，反向偏离产生负乘积；平均后得到协方差。

**第 3 步｜推广到多个变量。** 对中心化数据矩阵 $X_c$，

$$
\Sigma=\frac{1}{n-1}X_c^TX_c
$$

其中第 $(j,k)$ 个元素正是变量 $j$ 与 $k$ 的协方差。

**把上面的关系收束为本章计算式：**

$$
\operatorname{Cov}(X,Y)=\frac{1}{n-1}\sum_i(x_i-\bar{x})(y_i-\bar{y}),\qquad \Sigma_{jk}=\operatorname{Cov}(X_j,X_k)
$$

**符号解释：** 协方差矩阵 $\Sigma$ 汇总多个变量两两共同变化。

**代码对应：** 联合图用于深挖一对变量，成对图用于扫描多个变量；重点关系再用数值统计复核。

**使用边界：** 量纲会影响协方差，比较不同尺度变量时通常使用相关矩阵。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f'Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f'样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行'
)


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

grid = sns.jointplot(
    data=marketing,
    x="visits",
    y="sales",
    kind="scatter",
    height=6,
    joint_kws={"alpha": 0.45, "s": 24},
    color="#1a73e8",
)
grid.set_axis_labels("访问量", "销售额")
grid.fig.suptitle("访问量与销售额联合分布", y=1.02)
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**

把上面基础图表里的 kind 参数从 "scatter" 改成 "kde"，再运行一次，看看中心主图和边缘单变量分布的信息展示发生了什么变化。想一想：散点图更强调什么，KDE 密度图更强调什么，谁更适合大样本数据？

在下面的代码格里补全代码，最后一行为画出你修改后的联合分布图。


In [ ]:
try:
    # 请在下方填写代码
    # 目标：把基础图表的 kind 从 "scatter" 改成 "kde"，观察边缘分布的信息变化。
    kind = "scatter"  # TODO: 把散点改成核密度估计 "kde"

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

grid = sns.jointplot(
    data=marketing,
    x="visits",
    y="sales",
    hue="channel",
    height=7,
    palette="colorblind",
    joint_kws={"alpha": 0.45, "s": 24},
)
grid.set_axis_labels("访问量", "销售额")
grid.fig.suptitle("分渠道联合分布", y=1.02)
plt.show()


## 参数说明

- kind：scatter/hex/kde/reg/hist
- marginal_kws：边缘设置
- height：尺寸
- ratio：主图比例


## 结果解读

中心图读取关系，边缘图读取各变量分布；两部分应结合解释。


## 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 只看中心趋势忽略边缘偏态
- 大样本散点过度重叠
- 不适合一次比较很多变量


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把散点联合换成六边形密度，观察密集区的表达
    # 【目标】用六边形(hex)以颜色深浅表示密度，替代散点，适合大量样本。
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 起点示例(已可运行)：kind 从 scatter 换成 hex，用密度代替散点。
    grid = sns.jointplot(
        data=marketing,
        x="visits",
        y="sales",
        kind="hex",
        height=6,
        cmap="Blues",
    )
    grid.set_axis_labels("访问量", "销售额")
    grid.fig.suptitle("访问量与销售额六边形密度", y=1.02)
    plt.show()

    # ---- 反思记录：密度图看密集区，和散点图有何不同 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用jointplot同时展示两个变量关系及各自边缘分布。


### 你已经掌握

- 判断联合分布图（jointplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `kind` | scatter/hex/kde/reg/hist |
| `marginal_kws` | 边缘设置 |
| `height` | 尺寸 |
| `ratio` | 主图比例 |


### 需要注意

- 只看中心趋势忽略边缘偏态
- 大样本散点过度重叠
- 不适合一次比较很多变量


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 请在下方填写代码（参考答案）

kind = "kde"
grid = sns.jointplot(
    data=marketing,
    x="visits",
    y="sales",
    kind=kind,
    height=6,
    color="#1a73e8",
)
grid.set_axis_labels("访问量", "销售额")
grid.fig.suptitle("访问量与销售额 KDE 联合分布", y=1.02)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

grid = sns.jointplot(
    data=marketing,
    x="ad_spend",
    y="conversion",
    kind="hex",
    height=6,
    color="#188038",
)
grid.set_axis_labels("广告投入", "转化率")
grid.fig.suptitle("广告投入与转化率六边形密度", y=1.02)
plt.show()
